In [ ]:
# Install the latest Hugging Face libraries for Gemma inference
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate bitsandbytes

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import re, sys, json, time, random, shutil, tempfile, subprocess, glob
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

_hits = glob.glob("/kaggle/input/**/config.json", recursive=True)
assert _hits, "No model found in /kaggle/input - attach Gemma 4 via Add Input first."
MODEL_PATH = os.path.dirname(_hits[0])
print("Using model at:", MODEL_PATH)

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto", quantization_config=bnb)
model.eval()
print("Gemma loaded successfully!")
print("Model footprint GB:", round(model.get_memory_footprint() / 1e9, 1))
print("GPU in use GB:     ", round(torch.cuda.memory_allocated() / 1e9, 1))

USAGE = {"prompt_tokens": 0, "completion_tokens": 0}
SAMPLE = False

def reset_usage():
    USAGE["prompt_tokens"] = 0
    USAGE["completion_tokens"] = 0

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)

def llm(prompt, max_tokens=2048, temperature=0.7):
    """Send one prompt to Gemma and return ONLY the newly generated text."""
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    kwargs = dict(max_new_tokens=max_tokens)
    if SAMPLE:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        kwargs.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    USAGE["prompt_tokens"] += int(inputs["input_ids"].shape[-1])
    USAGE["completion_tokens"] += int(new_tokens.shape[-1])
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
import pathlib

FILE_FORMAT = """Output every file you create or change in exactly this format, with the COMPLETE file contents (no placeholders, no diffs):

### FILE: path/to/file.py
```python
<full file contents>
```

Do not output anything except these file blocks. Never modify files under tests/."""

FILE_RE = re.compile(r"###\s*FILE:\s*(\S+)\s*\n```[a-zA-Z]*\n(.*?)```", re.DOTALL)

def is_test_path(path):
    base = os.path.basename(path)
    return path.startswith("tests/") or base.startswith("test_") or base == "conftest.py"

def parse_files(text):
    """Turn the model's '### FILE:' blocks into {path: code}. Test files and unsafe paths are ignored."""
    files = {}
    for path, code in FILE_RE.findall(text):
        path = path.strip()
        while path.startswith("./"):
            path = path[2:]
        if path.startswith("/") or ".." in path.split("/"):
            continue
        if is_test_path(path):
            continue
        files[path] = code
    return files

def repo_to_text(repo):
    return "\n\n".join(f"### FILE: {p}\n```python\n{c}\n```" for p, c in repo.items())

def load_repo(folder, exts=(".py",)):
    """Read a cloned repo folder into {relative_path: contents}."""
    repo = {}
    for root, _, names in os.walk(folder):
        if any(skip in root for skip in (".git", "__pycache__", ".venv", "node_modules")):
            continue
        for n in names:
            if n.endswith(exts):
                full = os.path.join(root, n)
                repo[os.path.relpath(full, folder)] = open(full, encoding="utf-8", errors="ignore").read()
    return repo

ERR_RE = re.compile(r"\b(ImportError|ModuleNotFoundError|NameError|AttributeError|SyntaxError|IndentationError)\b")

def run_cmd(cmd, cwd, timeout=180):
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, p.stdout + p.stderr
    except subprocess.TimeoutExpired:
        return 124, "TIMEOUT"

def run_qa(repo, min_files=1):
    """Write the repo to a temp folder, compile it, run pytest. Returns hard numbers, not opinions."""
    workdir = tempfile.mkdtemp(prefix="qa_")
    try:
        for path, content in repo.items():
            full = os.path.join(workdir, path)
            os.makedirs(os.path.dirname(full) or workdir, exist_ok=True)
            with open(full, "w", encoding="utf-8") as f:
                f.write(content)
        rc_c, out_c = run_cmd([sys.executable, "-m", "compileall", "-q", "."], workdir)
        rc_t, out_t = run_cmd([sys.executable, "-m", "pytest", "-q", "--tb=short", "-p", "no:cacheprovider"], workdir)
        n_src = len([p for p in repo if p.endswith(".py") and not is_test_path(p)])
        struct = sum(1 for line in out_t.splitlines() if line.startswith("E ") and ERR_RE.search(line))
        struct += len(ERR_RE.findall(out_c))
        compiles, tests_pass = rc_c == 0, rc_t == 0
        return {
            "compiles": compiles,
            "tests_pass": tests_pass,
            "structural_errors": struct,
            "n_source_files": n_src,
            "success": compiles and tests_pass and n_src >= min_files,
            "report": (out_c + "\n" + out_t)[-2500:],
        }
    finally:
        shutil.rmtree(workdir, ignore_errors=True)

NO_PATCH_QA = {"compiles": False, "tests_pass": False, "structural_errors": 0, "n_source_files": 0,
               "success": False, "report": "No parseable '### FILE:' blocks were found in your output."}

In [ ]:
VERBOSE = True
def log(*a):
    if VERBOSE:
        print(*a)

def architect(task, repo):
    return llm(f"""ROLE: Architect. You are a staff software architect.

Task: {task}

Current codebase:
{repo_to_text(repo)}

Write a concise step-by-step refactoring plan. For each file to create or modify, say what goes in it and which public functions other files rely on. Existing tests in tests/ must keep passing unchanged. Do not write code yet.""")

def developer(task, repo, plan, prev_patch=None, feedback=None):
    retry = ""
    if prev_patch and feedback:
        retry = f"""
Your previous attempt:
{prev_patch}

QA feedback you must fix:
{feedback}
"""
    return llm(f"""ROLE: Developer. You are a senior engineer implementing an architect's plan.

Task: {task}

Current codebase:
{repo_to_text(repo)}

Architect's plan:
{plan}
{retry}
{FILE_FORMAT}""")

def qa_review(task, plan, patch_files):
    """Called only after the real tests already pass: checks the architecture goal itself."""
    return llm(f"""ROLE: QA reviewer. You are a strict code reviewer. All tests already pass.

Task: {task}

Architect's plan:
{plan}

Patched files:
{repo_to_text(patch_files)}

Check that the patch truly satisfies the task (clear separation of layers, no duplicated logic left behind, no missing imports).
If it does, reply with exactly the single word APPROVED. Otherwise reply with a short numbered list of required fixes.""")

def is_approved(review):
    return review.strip().upper().startswith("APPROVED")

def failure_reason(qa, accepted):
    """Classify why a run failed, for the results table."""
    if qa is NO_PATCH_QA or qa["n_source_files"] == 0:
        return "no_parseable_files"
    if not qa["compiles"]:
        return "compile_error"
    if not qa["tests_pass"]:
        return "tests_failed"
    if not qa["success"]:
        return "too_few_files"
    if not accepted:
        return "review_not_approved"
    return "ok"

def finish(method, qa, accepted, rounds, t0, patch, raw_outputs=None):
    reason = failure_reason(qa, accepted)
    return {
        "method": method, "accepted": accepted, "success": qa["success"] and accepted,
        "failure_reason": reason,
        "compiles": qa["compiles"], "tests_pass": qa["tests_pass"],
        "structural_errors": qa["structural_errors"], "n_source_files": qa["n_source_files"],
        "rounds": rounds,
        "prompt_tokens": USAGE["prompt_tokens"], "completion_tokens": USAGE["completion_tokens"],
        "total_tokens": USAGE["prompt_tokens"] + USAGE["completion_tokens"],
        "seconds": round(time.time() - t0, 1),
        "final_files": patch,
        "raw_outputs": raw_outputs or {},
    }

def run_committee(task, repo, min_files=1, max_rounds=3):
    """Architect plans -> Developer patches -> QA runs tests, then reviews. Loop until both approve."""
    reset_usage(); t0 = time.time()
    raw = {"architect": "", "rounds": []}
    log("[Architect] drafting plan...")
    plan = architect(task, repo)
    raw["architect"] = plan
    patch_text, feedback, patch, qa = None, None, {}, NO_PATCH_QA
    for rnd in range(1, max_rounds + 1):
        log(f"\n--- Consensus round {rnd} ---")
        round_log = {"round": rnd}
        log("[Developer] writing patch...")
        patch_text = developer(task, repo, plan, patch_text, feedback)
        round_log["developer_output"] = patch_text
        patch = parse_files(patch_text)
        if not patch:
            qa, feedback = NO_PATCH_QA, NO_PATCH_QA["report"]
            round_log["qa_result"] = "no_parseable_files"
            log("[QA] rejected: no parseable files")
            raw["rounds"].append(round_log)
            continue
        qa = run_qa({**repo, **patch}, min_files)
        round_log["qa_report"] = qa["report"]
        if not qa["success"]:
            feedback = "Automated checks failed:\n" + qa["report"]
            round_log["qa_result"] = failure_reason(qa, False)
            log(f"[QA] rejected: tests_pass={qa['tests_pass']} compiles={qa['compiles']} source_files={qa['n_source_files']}")
            raw["rounds"].append(round_log)
            continue
        log("[QA] tests pass, reviewing architecture...")
        review = qa_review(task, plan, {**repo, **patch})
        round_log["review"] = review
        if is_approved(review):
            round_log["qa_result"] = "approved"
            raw["rounds"].append(round_log)
            log("Consensus reached! Patch validated.")
            return finish("committee", qa, True, rnd, t0, patch, raw)
        feedback = "Reviewer feedback:\n" + review
        round_log["qa_result"] = "review_requested_changes"
        raw["rounds"].append(round_log)
        log("[QA] review requested changes")
    log("Max rounds reached without consensus.")
    return finish("committee", qa, False, max_rounds, t0, patch, raw)

def run_baseline(task, repo, min_files=1):
    """One Gemma call, no plan, no feedback loop — judged by exactly the same QA."""
    reset_usage(); t0 = time.time()
    text = llm(f"""ROLE: Developer (single pass). You are a senior software engineer.

Task: {task}

Current codebase:
{repo_to_text(repo)}

Existing tests in tests/ must keep passing unchanged.

{FILE_FORMAT}""")
    patch = parse_files(text)
    qa = run_qa({**repo, **patch}, min_files) if patch else NO_PATCH_QA
    raw = {"baseline_output": text}
    return finish("baseline", qa, qa["success"], 1, t0, patch, raw)

In [ ]:
# ========== TASK 1: payments_split ==========
payments_repo = {
    "server.py": '''import sqlite3

DB_PATH = "payments.db"

def handle_payment_request(request):
    db = sqlite3.connect(DB_PATH)
    cursor = db.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS ledger (id INTEGER PRIMARY KEY, amount REAL)")
    amount = request.get("amount")
    if amount is None or amount < 0:
        db.close()
        return "Error"
    cursor.execute("INSERT INTO ledger (amount) VALUES (?)", (amount,))
    db.commit()
    db.close()
    return "Success"
''',
    "tests/test_payments.py": '''import pathlib
import sqlite3
import sys

import pytest

ROOT = pathlib.Path(__file__).resolve().parent.parent


def rows():
    db = sqlite3.connect("payments.db")
    try:
        return db.execute("SELECT amount FROM ledger").fetchall()
    except sqlite3.OperationalError:
        return []
    finally:
        db.close()


@pytest.fixture
def app(tmp_path, monkeypatch):
    """Start the app fresh inside an empty working directory, like a real first launch."""
    for name, mod in list(sys.modules.items()):
        f = getattr(mod, "__file__", None)
        if f:
            p = pathlib.Path(f).resolve()
            if p.is_relative_to(ROOT) and p.parent != ROOT / "tests":
                del sys.modules[name]
    monkeypatch.chdir(tmp_path)
    import server
    return server


def test_valid_payment(app):
    assert app.handle_payment_request({"amount": 25.0}) == "Success"
    assert rows() == [(25.0,)]


def test_negative_rejected(app):
    assert app.handle_payment_request({"amount": -5}) == "Error"
    assert rows() == []


def test_missing_amount_rejected(app):
    assert app.handle_payment_request({}) == "Error"
''',
}

# ========== TASK 2: inventory_layers ==========
inventory_repo = {
    "inventory.py": '''import json
import os

DATA_FILE = "inventory.json"

def _load():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE) as f:
            return json.load(f)
    return {}

def _save(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f)

def add_item(name, quantity, price):
    if not name or quantity < 0 or price < 0:
        return {"ok": False, "error": "invalid input"}
    data = _load()
    if name in data:
        data[name]["quantity"] += quantity
    else:
        data[name] = {"quantity": quantity, "price": price}
    _save(data)
    return {"ok": True, "item": name}

def remove_item(name, quantity):
    data = _load()
    if name not in data:
        return {"ok": False, "error": "not found"}
    if data[name]["quantity"] < quantity:
        return {"ok": False, "error": "insufficient stock"}
    data[name]["quantity"] -= quantity
    if data[name]["quantity"] == 0:
        del data[name]
    _save(data)
    return {"ok": True, "remaining": data.get(name, {}).get("quantity", 0)}

def get_item(name):
    data = _load()
    if name not in data:
        return None
    return data[name]

def total_value():
    data = _load()
    return sum(item["quantity"] * item["price"] for item in data.values())

def handle_command(cmd):
    parts = cmd.strip().split()
    if not parts:
        return "error: empty command"
    action = parts[0].lower()
    if action == "add" and len(parts) == 4:
        result = add_item(parts[1], int(parts[2]), float(parts[3]))
        return f"added {parts[1]}" if result["ok"] else f"error: {result['error']}"
    elif action == "remove" and len(parts) == 3:
        result = remove_item(parts[1], int(parts[2]))
        return f"removed {parts[2]} of {parts[1]}" if result["ok"] else f"error: {result['error']}"
    elif action == "value":
        return f"total: {total_value()}"
    else:
        return "error: unknown command"
''',
    "tests/test_inventory.py": '''import pathlib
import sys
import pytest

ROOT = pathlib.Path(__file__).resolve().parent.parent

@pytest.fixture(autouse=True)
def fresh_app(tmp_path, monkeypatch):
    for name, mod in list(sys.modules.items()):
        f = getattr(mod, "__file__", None)
        if f:
            p = pathlib.Path(f).resolve()
            if p.is_relative_to(ROOT) and p.parent != ROOT / "tests":
                del sys.modules[name]
    monkeypatch.chdir(tmp_path)

@pytest.fixture
def inv():
    import inventory
    return inventory

def test_add_and_get(inv):
    result = inv.add_item("widget", 10, 2.50)
    assert result["ok"] is True
    item = inv.get_item("widget")
    assert item["quantity"] == 10
    assert item["price"] == 2.50

def test_add_increases_quantity(inv):
    inv.add_item("bolt", 5, 1.00)
    inv.add_item("bolt", 3, 1.00)
    assert inv.get_item("bolt")["quantity"] == 8

def test_remove_decreases(inv):
    inv.add_item("nut", 10, 0.50)
    result = inv.remove_item("nut", 4)
    assert result["ok"] is True
    assert inv.get_item("nut")["quantity"] == 6

def test_remove_all_deletes(inv):
    inv.add_item("washer", 3, 0.25)
    inv.remove_item("washer", 3)
    assert inv.get_item("washer") is None

def test_remove_insufficient(inv):
    inv.add_item("gear", 2, 5.00)
    result = inv.remove_item("gear", 10)
    assert result["ok"] is False
    assert inv.get_item("gear")["quantity"] == 2

def test_remove_not_found(inv):
    result = inv.remove_item("ghost", 1)
    assert result["ok"] is False

def test_invalid_input(inv):
    assert inv.add_item("", 1, 1.0)["ok"] is False
    assert inv.add_item("x", -1, 1.0)["ok"] is False
    assert inv.add_item("x", 1, -1.0)["ok"] is False

def test_total_value(inv):
    inv.add_item("a", 10, 2.00)
    inv.add_item("b", 5, 3.00)
    assert inv.total_value() == 35.00

def test_handle_command_add(inv):
    assert "added" in inv.handle_command("add screw 100 0.10")
    assert inv.get_item("screw")["quantity"] == 100

def test_handle_command_remove(inv):
    inv.add_item("nail", 50, 0.05)
    assert "removed" in inv.handle_command("remove nail 20")
    assert inv.get_item("nail")["quantity"] == 30

def test_handle_command_value(inv):
    inv.add_item("pipe", 4, 10.00)
    assert "40" in inv.handle_command("value")

def test_handle_command_unknown(inv):
    assert "error" in inv.handle_command("fly away")
''',
}

# ========== TASK 3: report_pipeline ==========
report_repo = {
    "reporter.py": '''"""
Monolithic report processor: parse, validate, aggregate, render — all in one function.
"""

def generate_report(raw_data, min_amount=0):
    """Process raw CSV-like sales data into a text report.

    raw_data: list of strings like "date,product,amount,region"
    min_amount: filter out rows below this amount
    Returns: dict with 'summary' (text), 'totals' (dict), 'valid_count' (int), 'invalid_count' (int)
    """
    # --- Parse ---
    records = []
    invalid_count = 0
    for line in raw_data:
        parts = line.strip().split(",")
        if len(parts) != 4:
            invalid_count += 1
            continue
        date, product, amount_str, region = parts
        try:
            amount = float(amount_str)
        except ValueError:
            invalid_count += 1
            continue
        records.append({"date": date, "product": product, "amount": amount, "region": region})

    # --- Validate ---
    valid = []
    for r in records:
        if r["amount"] < min_amount:
            continue
        if not r["product"].strip():
            invalid_count += 1
            continue
        valid.append(r)

    # --- Aggregate ---
    by_product = {}
    by_region = {}
    grand_total = 0
    for r in valid:
        by_product[r["product"]] = by_product.get(r["product"], 0) + r["amount"]
        by_region[r["region"]] = by_region.get(r["region"], 0) + r["amount"]
        grand_total += r["amount"]

    # --- Render ---
    lines = [f"Sales Report ({len(valid)} valid records, {invalid_count} invalid)"]
    lines.append(f"Grand Total: {grand_total:.2f}")
    lines.append("")
    lines.append("By Product:")
    for p in sorted(by_product):
        lines.append(f"  {p}: {by_product[p]:.2f}")
    lines.append("")
    lines.append("By Region:")
    for r in sorted(by_region):
        lines.append(f"  {r}: {by_region[r]:.2f}")

    return {
        "summary": "\\n".join(lines),
        "totals": {"by_product": by_product, "by_region": by_region, "grand_total": grand_total},
        "valid_count": len(valid),
        "invalid_count": invalid_count,
    }
''',
    "tests/test_reporter.py": '''import pathlib
import sys
import pytest

ROOT = pathlib.Path(__file__).resolve().parent.parent

@pytest.fixture(autouse=True)
def fresh_app(tmp_path, monkeypatch):
    for name, mod in list(sys.modules.items()):
        f = getattr(mod, "__file__", None)
        if f:
            p = pathlib.Path(f).resolve()
            if p.is_relative_to(ROOT) and p.parent != ROOT / "tests":
                del sys.modules[name]
    monkeypatch.chdir(tmp_path)

@pytest.fixture
def rpt():
    import reporter
    return reporter

SAMPLE_DATA = [
    "2026-01-01,Widget,100.00,North",
    "2026-01-02,Gadget,200.50,South",
    "2026-01-03,Widget,50.00,North",
    "2026-01-04,Gadget,75.25,East",
    "2026-01-05,,30.00,West",        # blank product -> invalid
    "bad line",                       # wrong columns -> invalid
    "2026-01-06,Doohickey,abc,North", # bad amount -> invalid
]

def test_valid_count(rpt):
    result = rpt.generate_report(SAMPLE_DATA)
    assert result["valid_count"] == 4

def test_invalid_count(rpt):
    result = rpt.generate_report(SAMPLE_DATA)
    assert result["invalid_count"] == 3

def test_grand_total(rpt):
    result = rpt.generate_report(SAMPLE_DATA)
    assert abs(result["totals"]["grand_total"] - 425.75) < 0.01

def test_by_product(rpt):
    result = rpt.generate_report(SAMPLE_DATA)
    prods = result["totals"]["by_product"]
    assert abs(prods["Widget"] - 150.00) < 0.01
    assert abs(prods["Gadget"] - 275.75) < 0.01

def test_by_region(rpt):
    result = rpt.generate_report(SAMPLE_DATA)
    regions = result["totals"]["by_region"]
    assert abs(regions["North"] - 150.00) < 0.01
    assert abs(regions["South"] - 200.50) < 0.01

def test_summary_text(rpt):
    result = rpt.generate_report(SAMPLE_DATA)
    assert "4 valid records" in result["summary"]
    assert "3 invalid" in result["summary"]
    assert "425.75" in result["summary"]

def test_min_amount_filter(rpt):
    result = rpt.generate_report(SAMPLE_DATA, min_amount=100)
    assert result["valid_count"] == 2
    assert abs(result["totals"]["grand_total"] - 300.50) < 0.01

def test_empty_input(rpt):
    result = rpt.generate_report([])
    assert result["valid_count"] == 0
    assert result["invalid_count"] == 0
    assert result["totals"]["grand_total"] == 0
''',
}

# ========== TASK 4: notifier_strategy ==========
notifier_repo = {
    "notifier.py": '''"""
Notification dispatcher with an if/elif chain.
"""

_sent_log = []  # in-memory log of sent notifications (for testing)

def clear_log():
    _sent_log.clear()

def get_log():
    return list(_sent_log)

def send_notification(channel, recipient, message):
    """Send a notification via the specified channel.

    channel: one of 'email', 'sms', 'push'
    recipient: email address, phone number, or device ID
    message: the notification text

    Returns dict with 'ok', 'channel', 'recipient', 'formatted_message'
    """
    if not recipient or not message:
        return {"ok": False, "error": "missing recipient or message"}

    if channel == "email":
        formatted = f"Subject: Notification\\nTo: {recipient}\\n\\n{message}"
    elif channel == "sms":
        truncated = message[:160]
        formatted = f"SMS to {recipient}: {truncated}"
    elif channel == "push":
        truncated = message[:256]
        formatted = f"PUSH[{recipient}]: {truncated}"
    else:
        return {"ok": False, "error": f"unknown channel: {channel}"}

    entry = {"channel": channel, "recipient": recipient, "formatted_message": formatted}
    _sent_log.append(entry)
    return {"ok": True, **entry}

def broadcast(channels, recipient, message):
    """Send the same message via multiple channels. Returns list of results."""
    return [send_notification(ch, recipient, message) for ch in channels]
''',
    "tests/test_notifier.py": '''import pathlib
import sys
import pytest

ROOT = pathlib.Path(__file__).resolve().parent.parent

@pytest.fixture(autouse=True)
def fresh_app(tmp_path, monkeypatch):
    for name, mod in list(sys.modules.items()):
        f = getattr(mod, "__file__", None)
        if f:
            p = pathlib.Path(f).resolve()
            if p.is_relative_to(ROOT) and p.parent != ROOT / "tests":
                del sys.modules[name]
    monkeypatch.chdir(tmp_path)

@pytest.fixture
def ntf():
    import notifier
    notifier.clear_log()
    return notifier

def test_email(ntf):
    r = ntf.send_notification("email", "a@b.com", "Hello")
    assert r["ok"] is True
    assert r["channel"] == "email"
    assert "Subject:" in r["formatted_message"]
    assert "a@b.com" in r["formatted_message"]
    assert "Hello" in r["formatted_message"]

def test_sms(ntf):
    r = ntf.send_notification("sms", "+1234567890", "Short msg")
    assert r["ok"] is True
    assert r["channel"] == "sms"
    assert "+1234567890" in r["formatted_message"]

def test_sms_truncates(ntf):
    long_msg = "x" * 300
    r = ntf.send_notification("sms", "+1234567890", long_msg)
    assert r["ok"] is True
    assert len(r["formatted_message"]) < 200

def test_push(ntf):
    r = ntf.send_notification("push", "device-abc", "Alert!")
    assert r["ok"] is True
    assert r["channel"] == "push"
    assert "device-abc" in r["formatted_message"]

def test_push_truncates(ntf):
    long_msg = "y" * 500
    r = ntf.send_notification("push", "dev-1", long_msg)
    assert r["ok"] is True
    assert len(r["formatted_message"]) < 300

def test_unknown_channel(ntf):
    r = ntf.send_notification("carrier_pigeon", "addr", "msg")
    assert r["ok"] is False

def test_missing_recipient(ntf):
    r = ntf.send_notification("email", "", "msg")
    assert r["ok"] is False

def test_missing_message(ntf):
    r = ntf.send_notification("sms", "+1234567890", "")
    assert r["ok"] is False

def test_log(ntf):
    ntf.send_notification("email", "a@b.com", "one")
    ntf.send_notification("sms", "+123", "two")
    log = ntf.get_log()
    assert len(log) == 2
    assert log[0]["channel"] == "email"
    assert log[1]["channel"] == "sms"

def test_clear_log(ntf):
    ntf.send_notification("push", "dev", "msg")
    ntf.clear_log()
    assert ntf.get_log() == []

def test_broadcast(ntf):
    results = ntf.broadcast(["email", "sms", "push"], "user@x.com", "Broadcast!")
    assert len(results) == 3
    assert all(r["ok"] for r in results)
    assert ntf.get_log().__len__() == 3
''',
}

# ========== TASK LIST ==========
TASKS = [
    {
        "name": "payments_split",
        "repo": payments_repo,
        "task": ("Refactor this monolith. Separate the database layer, the business-logic layer and the routing layer "
                 "into distinct files. Keep `server.handle_payment_request(request)` working as the public entry point."),
        "min_files": 3,
    },
    {
        "name": "inventory_layers",
        "repo": inventory_repo,
        "task": ("Refactor this monolithic inventory manager. Separate the JSON storage layer, the business-logic layer "
                 "(add/remove/get/total_value), and the command-parsing layer into distinct files. "
                 "Keep `inventory.handle_command(cmd)` working as the public entry point — it can delegate to the other layers."),
        "min_files": 3,
    },
    {
        "name": "report_pipeline",
        "repo": report_repo,
        "task": ("Refactor this monolithic report processor. Split `generate_report` into four separate stages in separate files: "
                 "a parser (CSV string -> records), a validator (filter invalid/below-threshold records), "
                 "an aggregator (compute totals by product and region), and a renderer (produce the summary text). "
                 "Keep `reporter.generate_report(raw_data, min_amount)` working as the public entry point — "
                 "it should call the four stages in sequence."),
        "min_files": 4,
    },
    {
        "name": "notifier_strategy",
        "repo": notifier_repo,
        "task": ("Refactor the if/elif notification dispatch into a strategy pattern. Create one class per channel "
                 "(EmailChannel, SmsChannel, PushChannel) in a separate file. The main notifier module should use a "
                 "registry/lookup to find the right channel class. Keep `notifier.send_notification(channel, recipient, message)` "
                 "and `notifier.broadcast(channels, recipient, message)` working as the public entry points."),
        "min_files": 2,
    },
    # --- add your real benchmark repos here ---
    # {
    #     "name": "my_repo",
    #     "repo": load_repo("/kaggle/working/repos/my-repo"),
    #     "task": "Describe the architectural change.",
    #     "min_files": 3,
    # },
]

# Sanity check: every original repo must pass its own tests
for t in TASKS:
    base = run_qa(t["repo"], min_files=1)
    print(f"{t['name']}: original tests_pass={base['tests_pass']}")
    if not base["tests_pass"]:
        print(base["report"])

In [ ]:
t = TASKS[0]
result = run_committee(t["task"], t["repo"], min_files=t["min_files"])

print("\n=== SUMMARY ===")
print({k: v for k, v in result.items() if k not in ("final_files", "raw_outputs")})
print("\n=== FINAL FILES ===")
for path, code in result["final_files"].items():
    print(f"\n### {path}\n{code}")

In [ ]:
import pandas as pd

N_RUNS = 1          # start with 1, raise to 3 once timing is confirmed
SAMPLE = True       # sampling makes repeated runs differ; seeds keep them reproducible
VERBOSE = False
RUN_DIR = "/kaggle/working/runs"

rows, artifacts = [], {}
for t in TASKS:
    for run in range(N_RUNS):
        for method in ("baseline", "committee"):
            set_seed(run)
            if method == "baseline":
                r = run_baseline(t["task"], t["repo"], t["min_files"])
            else:
                r = run_committee(t["task"], t["repo"], t["min_files"])

            # Save raw outputs and patch files to disk
            run_path = os.path.join(RUN_DIR, t["name"], method, f"run{run}")
            os.makedirs(run_path, exist_ok=True)
            with open(os.path.join(run_path, "log.json"), "w") as f:
                json.dump({k: v for k, v in r.items() if k not in ("final_files", "raw_outputs")}, f, indent=2)
            with open(os.path.join(run_path, "raw_outputs.json"), "w") as f:
                json.dump(r.get("raw_outputs", {}), f, indent=2, default=str)
            patch_dir = os.path.join(run_path, "patch")
            os.makedirs(patch_dir, exist_ok=True)
            for fname, code in r["final_files"].items():
                fpath = os.path.join(patch_dir, fname)
                os.makedirs(os.path.dirname(fpath) or patch_dir, exist_ok=True)
                with open(fpath, "w") as f:
                    f.write(code)

            artifacts[(t["name"], method, run)] = r.pop("final_files")
            r.pop("raw_outputs", None)
            r.update(task=t["name"], run=run)
            rows.append(r)
            pd.DataFrame(rows).to_csv("/kaggle/working/results.csv", index=False)
            print(f"{t['name']:20s} | run {run} | {method:9s} | {r['failure_reason']:22s} | tokens={r['total_tokens']:5d} | {r['seconds']}s")

df = pd.DataFrame(rows)
summary = df.groupby("method")[["success", "compiles", "tests_pass", "structural_errors", "rounds", "total_tokens", "seconds"]].mean().round(2)
print("\n=== BASELINE vs COMMITTEE (mean over all runs) ===")
print(summary)
summary.to_csv("/kaggle/working/summary.csv")

# Failure analysis
print("\n=== FAILURE ANALYSIS ===")
print(df.groupby(["method", "failure_reason"]).size().unstack(fill_value=0))